# 03 — Distillation Inspection

Compares teacher vs student: param count, mAP, FPS, qualitative predictions on a few test images.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from pathlib import Path
from ultralytics import YOLO
from src.modules.register import register_all
from src.paths import RUNS_DIR
register_all()

teacher_pt = RUNS_DIR/'combined_seed42'/'weights'/'best.pt'
student_pt = RUNS_DIR/'distill_seed42'/'weights'/'best.pt'
teacher = YOLO(str(teacher_pt))
student = YOLO(str(student_pt))
for name, m in [('teacher', teacher), ('student', student)]:
    n = sum(p.numel() for p in m.model.parameters())
    print(f'{name:8s} params={n/1e6:.2f}M')

In [ ]:
# Side-by-side predictions on 3 random test images
import random, cv2, matplotlib.pyplot as plt
from src.data.roboflow_pull import pull
dataset = pull()
test_images = list((dataset/'test'/'images').glob('*.*'))
random.seed(7)
picks = random.sample(test_images, 3)

fig, axes = plt.subplots(3, 2, figsize=(14, 14))
for i, p in enumerate(picks):
    for j, (label, model) in enumerate([('teacher', teacher), ('student', student)]):
        r = model.predict(str(p), imgsz=768 if label=='teacher' else 640, conf=0.35, verbose=False)[0]
        im = r.plot()[:, :, ::-1]
        axes[i, j].imshow(im); axes[i, j].set_title(f'{label} — {p.name}'); axes[i, j].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# FPS comparison
from src.eval.fps_bench import gpu_fps
for label, w, sz in [('teacher', teacher_pt, 768), ('student', student_pt, 640)]:
    print(label, gpu_fps(w, imgsz=sz, n_iters=100))